# Multimodal Image Analysis: OFA+CI+ViT Ensemble Opt

## Project Purpose

The primary goal of this project is to evaluate and understand the performance of three distinct models: One For All (OFA), CLIP Interrogator (CI), and Vision Transformer (ViT), each offering unique capabilities in processing and interpreting image data.

- **OFA**: Analyzed for its ability to generate descriptive text from images, highlighting its utility in tasks requiring image-to-text translation.
- **CI**: Evaluated for its proficiency in creating accurate and relevant text prompts based on image content, leveraging the strength of CLIP's vision-language understanding.
- **ViT**: Assessed for its effectiveness in transforming images into meaningful embeddings, showcasing its power in pure image understanding without text generation.

The culmination of this project is to create an ensemble model that synergizes the strengths of these three models. The ensemble is optimized to achieve a more accurate and holistic understanding of images, blending textual and visual information processing capabilities. This endeavor aims not only to leverage the individual model's strengths but also to enhance overall performance and accuracy in image interpretation tasks.

## Overview

This project explores the integration and optimization of three advanced machine learning models—OFA, CI, and ViT—to analyze and interpret handwritten math assignments. By leveraging the unique strengths of each model, we aim to develop a robust ensemble method that enhances the accuracy and reliability of image and text interpretation.
- Original Notebook: [CLIPInterrogator+OFA+ViT](https://www.kaggle.com/code/motono0223/clipinterrogator-ofa-vit)

### 1. One For All (OFA)
OFA is a multimodal model capable of handling various tasks across different domains. We introduce OFA, showcase its application on handwritten math assignments, and analyze its performance on a recognized image dataset to gauge its overall accuracy.
- Original Paper: [OFA: UNIFYING ARCHITECTURES, TASKS, AND MODALITIES
THROUGH A SIMPLE SEQUENCE-TO-SEQUENCE LEARNING
FRAMEWORK](https://arxiv.org/pdf/2202.03052.pdf)

- Original Notebook: [OFA Transformer [LB: 0.42644]](https://www.kaggle.com/code/mayukh18/ofa-transformer-lb-0-42644)

### 2. CLIP Interrogator (CI)
The CLIP (Contrastive Language–Image Pretraining) Interrogator is a cutting-edge tool that optimizes text prompts to correspond with visual content effectively. In this project, we introduce the CI's mechanism and demonstrate its capability to analyze handwritten math assignments. Furthermore, we evaluate its general accuracy using a known image dataset to understand its performance and reliability.
- Original Notebook: [[LB: 0.45836] ~ BLIP+CLIP | CLIP Interrogator](https://www.kaggle.com/code/leonidkulyk/lb-0-45836-blip-clip-clip-interrogator)

### 3. Visual Transformer (ViT)
ViT has revolutionized the field of computer vision by applying transformer models, traditionally used in NLP, to image classification tasks. We will examine ViT's architecture, demonstrate its effectiveness in processing handwritten math assignments, and report on its accuracy against a standardized image dataset.
- Original Notebook: [CLIPInterrogator+OFA+ViT](https://www.kaggle.com/code/motono0223/clipinterrogator-ofa-vit)

### 4. Ensemble Model
Combining the strengths of OFA, CI, and ViT, we create an ensemble model aimed at optimizing the interpretation of handwritten math assignments. This segment will demonstrate the ensemble's effectiveness, discuss the optimization process, and present the accuracy metrics on a known image dataset. The ensemble coefficients used to integrate the models will also be disclosed, underscoring the rationale behind their determination.
- Original Notebook: [CLIPInterrogator+OFA+ViT](https://www.kaggle.com/code/motono0223/clipinterrogator-ofa-vit)

By integrating these three models, this project seeks to enhance the analytical capabilities of machine learning in interpreting complex handwritten assignments, with a focus on mathematical content. Through empirical analysis and optimization, we aim to establish a reliable and accurate multimodal learning framework.

## Model Evaluation Approach

To assess the performance of our models, it's essential to understand that images are converted into numerical arrays, known as embeddings, during the processing phase. Although some models initially translate images into textual descriptions before embedding, ultimately, every model transforms the image into an embedding. This consistent endpoint allows for a unified method of evaluating model "accuracy" or, more precisely, the effectiveness of each model in capturing and representing the image's features numerically.

<img src="https://cdn-uploads.huggingface.co/production/uploads/noauth/sTYwsQTGxSd4753sonYkN.png" width="1000" alt="Image to Prompt to Embedding">

**Why only embeddings for some?**
- OFA and the CI can produce and display text because their operations include a stage of converting images to text. This text can be shown as part of the output.
- ViT, designed for image-to-embedding translation, lacks an intermediate text generation step, hence no textual output is directly available from it.
- In the ensemble of these models, since ViT contributes to the combined prediction without generating any text, the ensemble's output is also non-textual and focuses on optimized embedding representation rather than human-readable content.

## Summary of Results

In this project, we integrated three distinct models: OFA, CI, and ViT, to create a powerful ensemble for image analysis. By applying Linear Regression to optimize the ensemble weights (OFA: -0.0233, CI: 0.0201, ViT: 0.2164), significant improvements were achieved in the Mean Squared Error (MSE) performance of each individual model:

- **OFA**: Improved by 47%
- **CI**: Improved by 45%
- **ViT**: Improved by 39%

The optimized ensemble model achieved an impressive MSE of 0.0025, effectively transforming images into embeddings. This accomplishment sets the stage for future work aimed at developing a method to decode these embeddings back into human-readable text, thereby enhancing the interpretability of the model's output.

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        Imports <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## Imports

### General Imports

In [ ]:
# Install necessary package locally (assumed to be pre-downloaded)
!pip install -q /kaggle/input/stable-diffusion-data/transformers-4.18.0.dev0-py3-none-any.whl

In [ ]:
# Suppressing warnings to keep notebook output clean
import warnings
warnings.filterwarnings("ignore")

# Standard libraries for file and system operations
import os
import sys
import glob
from pathlib import Path

# Core data handling and scientific computing libraries
import numpy as np
import pandas as pd

# Visualization library
import matplotlib.pyplot as plt

# Machine learning utilities
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import normalize
from sklearn.linear_model import LinearRegression

# Image processing and deep learning libraries
from PIL import Image  # Python Imaging Library "Pillow" (Open, view, manipulate, and save images)
import torch  # PyTorch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Progress bar for loops in Jupyter Notebooks
from tqdm.notebook import tqdm

# Memory management
import gc # Garbage Collector

### OFA Imports

In [ ]:
# Importing from Hugging Face's Transformers library
from transformers import OFATokenizer, OFAModel  # For tokenizing and using the OFA model
from transformers.models.ofa.generate import sequence_generator  # For generating sequences with OFA

# Text processing utility
from gensim.parsing.preprocessing import remove_stopwords  # For removing common stopwords from text

# Importing Sentence Transformers for embedding textual data
# The sys.path.append line adds the sentence-transformers directory to the Python path, 
# allowing the subsequent import of SentenceTransformer and models
sys.path.append('../input/sentence-transformers-222/sentence-transformers')
from sentence_transformers import SentenceTransformer, models  # For text embedding and model utilities

# Setting up paths and initializing the Sentence Transformer model
st_model = SentenceTransformer('/kaggle/input/sentence-transformers-222/all-MiniLM-L6-v2')  # Loading the Sentence Transformer model

### CI Imports

In [ ]:
wheels_path = "/kaggle/input/clip-interrogator-wheels-x"
clip_interrogator_whl_path = f"{wheels_path}/clip_interrogator-0.4.3-py3-none-any.whl"

!pip install --no-index --find-links $wheels_path $clip_interrogator_whl_path -q

In [ ]:
# Python libraries for inspecting and reloading modules dynamically
import inspect  # For inspecting the source files of Python objects
import importlib  # For reloading modules after making changes

# Importing specific modules from the blip and clip_interrogator packages
from blip.models import blip  # BLIP model for image and text processing
from clip_interrogator import clip_interrogator  # Tool for generating text prompts from images using CLIP

# Importing the open_clip library, an open-source implementation of OpenAI's CLIP model
# open_clip provides a set of pre-trained CLIP models and utilities for image-text tasks
import open_clip

### VIT Imports

In [ ]:
# Importing the 'timm' library (PyTorch Image Models)
# timm is a collection of image models, layers, and utilities for PyTorch, often used for benchmarking, fine-tuning, or as feature extractors.
import timm

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        0. Setup <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 0. Setup

### 0.0 Configuration

In [ ]:
class CFG:
    device = "cuda"
    seed = 42
    embedding_length = 384
    
    sentence_model_path = "/kaggle/input/sentence-transformers-222/all-MiniLM-L6-v2"
    
    blip_model_path = "/kaggle/input/clip-interrogator-models-x/model_large_caption.pth"
    ci_clip_model_name = "ViT-H-14/laion2b_s32b_b79k"
    clip_model_name = "ViT-H-14"
    clip_model_path = "/kaggle/input/clip-interrogator-models-x/CLIP-ViT-H-14-laion2B-s32B-b79K/open_clip_pytorch_model.bin"
    ci_cache_path = "/kaggle/input/clip-interrogator-models-x"
    
    vit_model_path = '/kaggle/input/k/millerrfu/stable-diffusion-vit-baseline-train/vit_base_patch16_224.pth'
    vit_model_name = 'vit_base_patch16_224'
    vit_input_size = 224
    vit_batch_size = 64

In [ ]:
comp_path = Path('/kaggle/input/stable-diffusion-image-to-prompts/')  # Path to the competition dataset
df_submission = pd.read_csv(comp_path / 'sample_submission.csv', index_col='imgId_eId')
df_prompts = pd.read_csv(comp_path / 'prompts.csv', index_col='imgId')

# Code from competition organizers
images = os.listdir(comp_path / 'images')
imgIds = [i.split('.')[0] for i in images] # remove ".png" from image name

eIds = list(range(CFG.embedding_length))

imgId_eId = [
    '_'.join(map(str, i)) for i in zip(
        np.repeat(imgIds, CFG.embedding_length),
        np.tile(range(CFG.embedding_length), len(imgIds))
    )
]

assert sorted(imgId_eId) == sorted(df_submission.index)

In [ ]:
def Display_Prompts(in_prompts, in_head=5):
    return print(f"Size: {len(in_prompts)}", "\nHead:", *in_prompts[:in_head], sep='\n')

def Display_Embeddings(in_embeddings, in_head=5):
    return print(f"Size: {in_embeddings.shape[0]}", "\nHead:", *in_embeddings[:in_head], sep='\n')


default = df_submission.iloc[:, 0].to_numpy() # true embeddings

def Get_MSE(in_pred, in_true=default):
    return mean_squared_error(in_true, in_pred[-len(in_true):]) # 2688

def Display_MSE(in_mse):
    return print("Mean Squared Error:", in_mse)

In [ ]:
# Retrieving the list of file paths
g = glob.glob("/kaggle/input/stable-diffusion-image-to-prompts/images/*")
im_mhw_paths = ["/kaggle/input/mathhw/MathHW.png"] + g

## 0.1 True Embeddings

#### 0.1 (a) True Prompts

In [ ]:
stable_diff_prompts = df_prompts.iloc[:, 0].tolist()

stable_diff_mhw_prompts = ["[hand-drawn, not generated by Stable Diffusion]"] + stable_diff_prompts
Display_Prompts(stable_diff_mhw_prompts)

#### 0.1 (b) Observing the Embeddings

In [ ]:
true_e = default.copy()
Display_Embeddings(true_e)

### 0.2 True Values

In [ ]:
def Embeddings_to_Captions(in_embeddings=true_e, in_known_ub=8, in_head=5):
    n = in_embeddings.shape[0] // 384
    capts = [''] + [', '.join([str(x) for x in in_embeddings[384*i:384*i+in_head]])+', ...' for i in range(n)]
    return capts[-in_known_ub:]

true_e_caps = Embeddings_to_Captions()

# Displaying the images and their Stable Diffusion prompts

blanks = ['']*len(im_mhw_paths)

def Plot(in_pred_embeddings=blanks,
         in_mse=0,
         in_pred_prompts=blanks,
         in_embeddings=true_e_caps,
         in_prompts=stable_diff_mhw_prompts,
         in_im_paths=im_mhw_paths
        ):
    
    if in_pred_embeddings != blanks:
        in_pred_embeddings = Embeddings_to_Captions(in_pred_embeddings)
    
    if in_mse != 0:
        Display_MSE(in_mse)
        print()
    
    n = len(in_im_paths)
    
    fig, ax = plt.subplots(n, 1, figsize=(4, n*5))

    for i,im_path in enumerate(im_mhw_paths):
        # Open the image
        image = Image.open(im_path)

        # display the image on the ith subplot
        ax[i].imshow(image)
        
        space = 2
        check1 = int(in_prompts[i]!='')
        check2 = int(in_pred_prompts[i]!='')
        check3 = int(in_embeddings[i]!='')
        check4 = int(in_pred_embeddings[i]!='')
        captions = [
            f"Stable Diffusion Prompt:\n{in_prompts[j]}"*check1 + "\n"*space +
            f"Predicted Prompt:\n{in_pred_prompts[j]}"*check2 + "\n"*space +
            f"True Embeddings:\n{in_embeddings[j]}"*check3 + "\n"*space +
            f"Predicted Embeddings:\n{in_pred_embeddings[j]}"*check4
                    for j in range(n)
                   ]

        # adds a text caption on the ith subplot
        ax[i].text(1.1, .5, captions[i], horizontalalignment='left', verticalalignment='center', transform=ax[i].transAxes)

Plot()

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        1. One For All (OFA) <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 1. One For All (OFA)

OFA is a unified framework for multimodal pretraining designed to simplify complex task and modality-specific processes. It is a task-agnostic and modality-agnostic system that encompasses a wide range of both cross-modal and unimodal tasks within a straightforward sequence-to-sequence learning approach. Utilizing instruction-based learning, OFA eliminates the need for additional task-specific layers in both pretraining and finetuning phases. Despite being pretrained on a relatively modest dataset of 20 million image-text pairs, it sets new benchmarks in various cross-modal tasks and competes strongly in unimodal tasks. Additionally, OFA demonstrates effective adaptability to new tasks and domains, with resources [available publicly](https://github.com/OFA-Sys/OFA) for further research and development.

### 1.0 Pretrained OFA Model

In [ ]:
ckpt_dir = "/kaggle/input/stable-diffusion-data/OFA-large-caption/"
image_dir = "/kaggle/input/stable-diffusion-image-to-prompts/images"

batch_size = 24

In [ ]:
# Source: https://huggingface.co/OFA-Sys/ofa-large

# Used for image normalization
mean, std = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]

# Desired resolution for image resizing
resolution = 480

# Chains together multiple data transformations
# 1: Converts the input image into RGB format
# 2: Resizes the image to desired resolution
# 3: Converts image to a PyTorch tensor
# 4: Normalizes the pixel values using the image normalization parameters from above
patch_resize_transform = transforms.Compose([
        lambda image: image.convert("RGB"),
        transforms.Resize((resolution, resolution), interpolation=Image.BICUBIC),
        transforms.ToTensor(), 
        transforms.Normalize(mean=mean, std=std)
    ])

# Customer tokenizer and custom model
# OFA is a unified multimodal pretrained model that unifies modalities and tasks 
# to a simple sequence-to-sequence learning framework.
tokenizer = OFATokenizer.from_pretrained(ckpt_dir)
model = OFAModel.from_pretrained(ckpt_dir, use_cache=False).cuda()
txt = " what does the image describe?"

# The tokenized input text
inputs = tokenizer([txt], return_tensors="pt").input_ids

### 1.1 OFA Embeddings

#### 1.1 (a) OFA Prompts

In [ ]:
# Predicting the prompts

def OFA_Prompts(in_im_paths=im_mhw_paths):
    pred_prompts = []
    for i,impath in enumerate(im_mhw_paths):
        # Opens the image
        image = Image.open(impath)
        
        # Uses the data transformation function above
        # sends the tensor to GPU and creates a batch of size 1
        image_t = patch_resize_transform(image).cuda().unsqueeze(0)
        
        # Generates text captions for the input image
        # - tokenized input text and image are passed in (both are sent to GPU)
        # - num_beams: controls the beam search
        # - no_repeat_ngram_size: controls n-gram repetition during text generation
        pred = model.generate(inputs.cuda(), patch_images=image_t.cuda(), num_beams=5, no_repeat_ngram_size=2)
        
        # Decodes the generated text captions from the model's output tensor
        # skip_special_tokens: start-of-sentence and end-of-sentence tokens should be skipped
        pred_prompts.append(tokenizer.batch_decode(pred, skip_special_tokens=True))
    return pred_prompts

ofa_prompts_full = OFA_Prompts()
ofa_prompts = [x[0].strip() for x in ofa_prompts_full]
# temp = ofa_prompts.copy()
# print(*temp, sep='\n')
Display_Prompts(ofa_prompts)

#### 1.1 (b) Encoding the OFA Prompts

In [ ]:
# Generates image batches from a directory of image files
class ImageGen(Dataset):
    def __init__(self, roots, batch_size=32, transform=None):
        self.roots = roots if isinstance(roots, list) else [roots]
        self.transform = transform
        self.im_paths = []
        for root in self.roots:
            self.im_paths.extend([os.path.join(root, f) for f in os.listdir(root) if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.batch_size = batch_size
        
        # Total number of images
        self.sz = len(self.im_paths)
        
        # Total number of batches to generate
        self.genlen = self.sz//self.batch_size + int(self.sz%self.batch_size > 0)
        
    # Retrieves an item from the dataset at a specific index
    # Returns a tuple containing the image batch and respective file IDs
    def __getitem__(self, index):
        
        # Checks if the given index is within bounds 
        if index >= self.genlen:
            raise IndexError("Out of bounds")
        
        # Calculates start and end indices of image file paths using batch size and given index
        l, r = index*self.batch_size, min(self.sz, (index+1)*self.batch_size)
        
        # Creates a list of file paths for the current batch
        f_paths = self.im_paths[l:r]
        
        # Creates a list of file IDs by removing the file extension from the image file paths
        f_ids = [os.path.basename(path)[:-4] for path in f_paths]
        
        # Same as above
        ims = [Image.open(f_path) for f_path in f_paths]
        ims = [patch_resize_transform(im).cuda().unsqueeze(0) for im in ims]
        
        # Creates the final image batch tensor
        ims = torch.cat(ims)
        
        # Returns the image batch tensor and the file IDs, as a tuple
        return ims, f_ids
    
    # Returns the total # of batches to generate (which includes the entire length of the dataset)
    def __len__(self):
        return self.genlen

In [ ]:
def OFA_Embeddings(in_prompts=ofa_prompts_full):
    
    # Will store caption IDs and corresponding embeddings
    sub_ids = []
    sub_embeds = []
    
    # Uses the ImageGen class from above
    imgen = ImageGen(["/kaggle/input/mathhw", image_dir], batch_size)
    
    # Iterates over the batches of image tensors
    for b in imgen:
        
        # Iterates over the file IDs in the current batch
        for j in range(len(b[1])):

            # 384 caption IDs for a given file ID
            sub_ids.extend([f"{b[1][j]}_{i}" for i in range(384)])
        
        img_batch = b[0]
        
        pred = model.generate(inputs.repeat(len(img_batch), 1).cuda(), patch_images=img_batch, num_beams=5, no_repeat_ngram_size=2)
        
        # Decodes the generated captions from the model output tensor
        pred_prompts = tokenizer.batch_decode(pred, skip_special_tokens=True)

        # Removes common stopwords from the generated captions
        pred_prompts = [remove_stopwords(text) for text in pred_prompts]
        
        # Encodes the cleaned captions into embeddings using a pre-trained sentence-transformers model
        # Flattens the resulting embeddings into a 1-D tensor
        embeddings = st_model.encode(pred_prompts).flatten()
        
        # Stores the embeddings of the corresponding captions
        sub_embeds.extend(embeddings)
    return np.array(sub_embeds)
    
ofa_pred = OFA_Embeddings()
Display_Embeddings(ofa_pred)

### 1.2 OFA MSE

In [ ]:
ofa_mse = Get_MSE(ofa_pred)
Display_MSE(ofa_mse)

### 1.3 OFA Results

In [ ]:
Plot(ofa_pred, ofa_mse, ofa_prompts)

### Freeing Up Memory

In [ ]:
del model, tokenizer, st_model
torch.cuda.empty_cache()
gc.collect()

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        2. CLIP Interrogator (CI) <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 2. CLIP Interrogator (CI)

### CI Overview

CI, a tool engineered to optimize text prompts, integrates OpenAI's [CLIP](https://openai.com/blog/clip/) with Salesforce's [BLIP](https://blog.salesforceairesearch.com/blip-bootstrapping-language-image-pretraining/), enhancing image-text matching.

<br>

**Pipeline Visual**

For a detailed view of its workflow, see: [Diversify photo database with Clip Interrogator](https://medium.com/@silkworm/diversify-photo-database-with-clip-interrogator-5dd1833be9f5)

![CLIP Interrogator Pipeline](https://user-images.githubusercontent.com/45982614/220214422-19529ba3-9c13-40cd-a3a6-434785002974.png)

**Explore CLIP Interrogator on Hugging Face Space**

Interested in exploring CLIP Interrogator? Check it out on [Hugging Face Space](https://huggingface.co/spaces/pharma/CLIP-Interrogator). Here's what it looks like in action:

<img src="https://cdn-uploads.huggingface.co/production/uploads/noauth/5scnD9MoeoHn8ftLu_801.png" width="1000" alt="Example Output">

### Pretrained CI Model

In [ ]:
# replace tokenizer path to prevent downloading
blip_path = inspect.getfile(blip)

fin = open(blip_path, "rt")
data = fin.read()

# uses the predownloaded tokenizer, instead of one from the internet
data = data.replace(
    "BertTokenizer.from_pretrained('bert-base-uncased')", 
    "BertTokenizer.from_pretrained('/kaggle/input/clip-interrogator-models-x/bert-base-uncased')"
)
fin.close()

fin = open(blip_path, "wt")

# updates the blip_path with the predownloaded tokenizer
fin.write(data)
fin.close()

# reload module
importlib.reload(blip)

In [ ]:
# Fix clip_interrogator bug
clip_interrogator_path = inspect.getfile(clip_interrogator.Interrogator)

fin = open(clip_interrogator_path, "rt")
data = fin.read()
data = data.replace(
    'open_clip.get_tokenizer(clip_model_name)', 
    'open_clip.get_tokenizer(config.clip_model_name.split("/", 2)[0])'
)
fin.close()

fin = open(clip_interrogator_path, "wt")
fin.write(data)
fin.close()

# reload module
importlib.reload(clip_interrogator)

In [ ]:
# Prepare CI

# Define CI configuration
model_config = clip_interrogator.Config(clip_model_name=CFG.ci_clip_model_name)
model_config.cache_path = CFG.ci_cache_path

# Define BLIP model
configs_path = os.path.join(os.path.dirname(os.path.dirname(blip_path)), 'configs')
med_config = os.path.join(configs_path, 'med_config.json')
blip_model = blip.blip_decoder(
    pretrained=CFG.blip_model_path,
    image_size=model_config.blip_image_eval_size, 
    vit=model_config.blip_model_type, 
    med_config=med_config
)
blip_model.eval()
blip_model = blip_model.to(model_config.device)
model_config.blip_model = blip_model

# Define the CLIP model
clip_model = open_clip.create_model(CFG.clip_model_name, precision='fp16' if model_config.device == 'cuda' else 'fp32')
open_clip.load_checkpoint(clip_model, CFG.clip_model_path)
clip_model.to(model_config.device).eval()
model_config.clip_model = clip_model

clip_preprocess = open_clip.image_transform(
    clip_model.visual.image_size,
    is_train = False,
    mean = getattr(clip_model.visual, 'image_mean', None),
    std = getattr(clip_model.visual, 'image_std', None),
)
model_config.clip_preprocess = clip_preprocess

In [ ]:
# Create CI object
ci = clip_interrogator.Interrogator(model_config)

In [ ]:
# Define interrogate function
# Original CI uses image_features and text_embeds matrix multiplication to fine the similarity between the corresponding image and text label
# It was found that using cosine similarity is much faster and the resulting score is almost identical

# Get labels embeddings
# Cosine similarity will be calculated across rows
cos = torch.nn.CosineSimilarity(dim=1)

# Stacks tensors converted from numpy array (t was the numpy array)
mediums_features_array = torch.stack([torch.from_numpy(t) for t in ci.mediums.embeds]).to(ci.device)
movements_features_array = torch.stack([torch.from_numpy(t) for t in ci.movements.embeds]).to(ci.device)
flavors_features_array = torch.stack([torch.from_numpy(t) for t in ci.flavors.embeds]).to(ci.device)

# Create main interrogation function
# It's modified version of the original nterrogate_classic method (https://github.com/pharmapsychotic/clip-interrogator/blob/main/clip_interrogator/clip_interrogator.py#L213)

def interrogate(image: Image) -> str:
    caption = ci.generate_caption(image)
    image_features = ci.image_to_features(image)
    
    # Grabs the top 1 image feature, text feature with the highesst cosine similarity for each text feature
    # text feature: medium, movement, flaves
    medium = [ci.mediums.labels[i] for i in cos(image_features, mediums_features_array).topk(1).indices][0]
    movement = [ci.movements.labels[i] for i in cos(image_features, movements_features_array).topk(1).indices][0]
    flaves = ", ".join([ci.flavors.labels[i] for i in cos(image_features, flavors_features_array).topk(3).indices])
    
    if caption.startswith(medium):
        prompt = f"{caption}, {movement}, {flaves}"
    else:
        prompt = f"{caption}, {medium}, {movement}, {flaves}"
    
    return clip_interrogator._truncate_to_fit(prompt, ci.tokenize)

### 2.1 CI Embeddings

#### 2.1 (a) CI Prompts

In [ ]:
def CI_Prompts(in_im_paths=im_mhw_paths):
    prompts = []
    for im in in_im_paths:
        img = Image.open(im).convert("RGB")
        generated = interrogate(img)
        prompts.append(generated)
    return prompts

ci_prompts = CI_Prompts()
Display_Prompts(ci_prompts)

#### 2.1 (b) Encoding the CI Prompts

In [ ]:
# Load the embedding model
st_model = SentenceTransformer(CFG.sentence_model_path)

ci_pred = st_model.encode(ci_prompts).flatten()
Display_Embeddings(ci_pred)

### 2.2 CI MSE

In [ ]:
ci_mse = Get_MSE(ci_pred)
Display_MSE(ci_mse)

### 2.3 CI Results

In [ ]:
Plot(ci_pred, ci_mse, ci_prompts)

### Freeing Up Memory

In [ ]:
del ci
del blip_model, clip_model
del st_model
torch.cuda.empty_cache()
gc.collect()

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        3. Vision Transformer (ViT) <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 3. Vision Transformer (ViT)

The ViT-B-16 model (ViT) represents a significant shift in computer vision, applying the principles of transformer architectures, primarily used in natural language processing, to image analysis. ViT divides an image into fixed-size patches, linearly embeds each of them, and processes these embeddings through a standard transformer encoder sequence. This approach allows ViT to capture complex patterns and relationships within the image data at various scales. Unlike traditional convolutional neural networks that process images in a hierarchical manner, ViT treats the image as a sequence of patches, enabling it to leverage the powerful self-attention mechanism to understand the global context of the image, leading to highly effective image classification and analysis performances.

### 3.0 Pretrained ViT Model

In [ ]:
class DiffusionTestDataset(Dataset):
    def __init__(self, images, transform):
        self.images = images
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Open the image file at the given index
        image = Image.open(self.images[idx])
        
        # Convert the image to RGB format
        image = image.convert('RGB')
        
        # Apply the specified transformations to the image
        image = self.transform(image)
        return image

images = [Path(x) for x in im_mhw_paths]

### 3.1 ViT Embeddings

In [ ]:
def ViT_Embeddings(
    images,
    model_path,
    model_name,
    input_size,
    batch_size
):
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    transform = transforms.Compose([
        transforms.Resize((input_size, input_size)), #, interpolation=Image.BICUBIC),
        transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomRotation(degrees=10),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    # Uses the DiffusionTestDataset class from above
    dataset = DiffusionTestDataset(images, transform)
    dataloader = DataLoader(
        dataset=dataset,
        shuffle=False,
        batch_size=batch_size,
        pin_memory=True,
        num_workers=2,
        drop_last=False
    )
    
    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=384
    )
    
    state_dict = torch.load(model_path)
    
    # List of keys to remove
    keys_to_remove = ["att.attention.0.weight", "att.attention.0.bias", 
                      "att.attention.1.weight", "att.attention.1.bias", 
                      "att.attention.3.weight", "att.attention.3.bias"]
    
    for key in keys_to_remove:
        if key in state_dict:
            del state_dict[key]
    
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    # Test time augmentation
    tta_preds = None
    for _ in range(2):
        preds = []
        for X in dataloader:
            X = X.to(device)
            with torch.no_grad():
                X_out = model(X).cpu().numpy()
                # L2 normalize -- Start
                X_out = X_out / ( np.abs(X_out).max(axis=-1, keepdims=True) + 0.0000001)  # To avoid to overflow at normalize()
                X_out = normalize( X_out )
                # L2 normalize -- End
                preds.append(X_out)
                
        if tta_preds is None:
            tta_preds = np.vstack(preds).flatten()
        else:
            tta_preds += np.vstack(preds).flatten()
    
    return tta_preds / 2

vit_pred = ViT_Embeddings(images, CFG.vit_model_path, CFG.vit_model_name, CFG.vit_input_size, CFG.vit_batch_size)
Display_Embeddings(vit_pred)

### 3.2 ViT MSE

In [ ]:
vit_mse = Get_MSE(vit_pred)
Display_MSE(vit_mse)

### 3.3 ViT Results

In [ ]:
Plot(vit_pred, vit_mse)

### Freeing Up Memory

In [ ]:
torch.cuda.empty_cache()
gc.collect()

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        4. Ensemble Model <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 4. Ensemble Model

### 4.0 Optimizing the Ensemble Model

In [ ]:
lt = true_e.shape[0] # 2688

# Stacking predictions as features
X = np.column_stack((ofa_pred[-lt:], ci_pred[-lt:], vit_pred[-lt:]))

# Fitting the linear model
model = LinearRegression().fit(X, true_e)

# Getting the optimal weights
optimal_weights = model.coef_
print("Optimal Weights:", *optimal_weights)

### 4.1 Ensemble Embeddings

In [ ]:
# Combine the predictions using the optimal weights

X_full = np.column_stack((ofa_pred, ci_pred, vit_pred))
optimal_pred = np.dot(X_full, optimal_weights)

Display_Embeddings(optimal_pred)

### 4.2 Ensemble MSE

In [ ]:
# Calculate the mean squared error
optimal_mse = Get_MSE(optimal_pred)

print()
print('Optimal Result')
Display_MSE(optimal_mse)
print()

print('Previous Results (OFA, CI, ViT)')
for mse in [ofa_mse, ci_mse, vit_mse]:
    Display_MSE(mse)

In [ ]:
# Calculate the percentage improvement
improvement_1 = ((ofa_mse - optimal_mse) / ofa_mse) * 100
improvement_2 = ((ci_mse - optimal_mse) / ci_mse) * 100
improvement_3 = ((vit_mse - optimal_mse) / vit_mse) * 100

print(f'Improvement for OFA: {improvement_1:.2f}%')
print(f'Improvement for CI:  {improvement_2:.2f}%')
print(f'Improvement for ViT: {improvement_3:.2f}%')

### 4.3 Ensemble Results

In evaluating the ensemble model, we observe significant improvements in performance as measured by Mean Squared Error (MSE). Below is the percentage improvement over the individual models:
- Improvement for OFA: 47%
- Improvement for CI: 45%
- Improvement for ViT: 39%

These improvements underscore the effectiveness of combining these models into an ensemble, optimizing their strengths to achieve a more accurate prediction outcome.

In [ ]:
Plot(optimal_pred, optimal_mse)

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 30px 5px; border-bottom: 15px solid #cccccc; text-align: center">
    <span style="font-size: 24px; font-weight: bold; background-color: #cccccc; padding: 0px 20px; color: #000000; border-radius: 5px;">
        5. Future Work <!-- Replace with your section title -->
    </span>
</div>

<br>
<br>

## 5. Future Work

A key objective for future exploration in this project is to develop a method for converting embeddings back into text. This would involve creating a mechanism to extract descriptive text from the ViT and the ensemble model's embeddings. Achieving this would bridge the gap between visual understanding and textual interpretation, allowing for a more comprehensive analysis of images and enhancing the interpretability of the models' outputs.

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

<!-- Section Separator -->
<div style="margin: 60px 10px; border-bottom: 30px solid #cccccc; text-align: center">
    <span style="font-size: 48px; font-weight: bold; background-color: #cccccc; padding: 0px 40px; color: #000000; border-radius: 10px;">
        Thank You! <!-- Replace with your section title -->
    </span>
</div>